# 📚 Knowledge Bases & RAG for Threat Intelligence

**Goal:** Give agents access to your entire organization's threat intel library, IR playbooks, and SIEM runbooks — grounded in your own documents.

```
  IR Playbooks ──┐
  Threat Intel ──┼──► Vector Store ──► FileSearchTool ──► Agent response
  SIEM Runbooks ─┘                         ▲                 with citations
                                           │
  Enterprise SIEM ──►  Azure AI Search ───┘  (large-scale OneLake)
```

**Two tools, different scales:**
- `FileSearchTool` — up to 10,000 files, built-in vector store, ideal for docs < 100MB
- `AzureAISearchTool` — enterprise scale, existing indices, OneLake / Sentinel Logs

In [ ]:
!uv pip install "azure-ai-projects>=2.0.0" azure-identity python-dotenv pandas matplotlib seaborn -q


In [ ]:
import os, json
from pathlib import Path
from dotenv import load_dotenv
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np
import seaborn as sns

load_dotenv(Path('..') / '.env', override=False)

AZURE_AI_PROJECT_ENDPOINT = os.getenv('AZURE_AI_PROJECT_ENDPOINT', '')
MODEL_DEPLOYMENT_NAME = os.getenv('MODEL_DEPLOYMENT_NAME', 'gpt-4o')
AZURE_AI_SEARCH_CONNECTION_NAME = os.getenv('AZURE_AI_SEARCH_CONNECTION_NAME', '')
AZURE_AI_SEARCH_INDEX_NAME = os.getenv('AZURE_AI_SEARCH_INDEX_NAME', 'soc-threat-intel')

MOCK_MODE = not bool(AZURE_AI_PROJECT_ENDPOINT)
SEARCH_ENABLED = bool(AZURE_AI_SEARCH_CONNECTION_NAME)
print('⚠️  MOCK MODE' if MOCK_MODE else f'✅ Connected: {AZURE_AI_PROJECT_ENDPOINT[:50]}...')
print('🔍 Azure AI Search:', '✅ Configured' if SEARCH_ENABLED else '⚠️  Not configured (set AZURE_AI_SEARCH_CONNECTION_NAME)')


## 🔵 Our Knowledge Base Files

We have two curated documents for our SOC knowledge base:
1. **IR Playbook** — step-by-step response for credential attacks
2. **Threat Intel Reference** — known bad actors, IOCs, MITRE ATT&CK quick reference

In [ ]:
data_dir = Path('data')
playbook_path = data_dir / 'ir_playbook_credential_attack.md'
threat_intel_path = data_dir / 'threat_intel_reference.md'

doc_stats = []
for p in [playbook_path, threat_intel_path]:
    content = p.read_text(encoding='utf-8')
    lines = content.splitlines()
    words = len(content.split())
    # rough chunk estimate (1 chunk ≈ 400 tokens ≈ 300 words)
    est_chunks = max(1, words // 300)
    sections = [l for l in lines if l.startswith('#')]
    doc_stats.append({
        'File': p.name,
        'Lines': len(lines),
        'Words': words,
        'Sections': len(sections),
        'Est. Chunks': est_chunks,
        'Size (KB)': round(len(content.encode()) / 1024, 1)
    })

df_docs = pd.DataFrame(doc_stats)
display(df_docs.style.bar(subset=['Words', 'Est. Chunks'], color='#4ecdc488')
        .set_properties(**{'background-color': '#1e2127', 'color': '#e6e6e6'}))

# Preview first 20 lines of playbook
print('\n📖 IR Playbook preview:')
print('─' * 50)
for line in playbook_path.read_text().splitlines()[:20]:
    print(line)

In [ ]:
# Visualize document structure
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
fig.patch.set_facecolor('#0d1117')

ax1 = axes[0]
ax1.set_facecolor('#0d1117')
bar_colors = ['#ff6b6b', '#4ecdc4']
bars = ax1.bar(df_docs['File'], df_docs['Words'], color=bar_colors, edgecolor='#333')
for bar, val in zip(bars, df_docs['Words']):
    ax1.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 5, str(val),
             ha='center', color='white', fontsize=10)
ax1.set_title('Document Size (Words)', color='white')
ax1.tick_params(colors='white')
ax1.set_xticklabels([f.replace('.md', '') for f in df_docs['File']], rotation=12, color='white', fontsize=8)
for s in ax1.spines.values(): s.set_color('#333')

ax2 = axes[1]
ax2.set_facecolor('#0d1117')
docs = df_docs['File'].tolist()
chunks = df_docs['Est. Chunks'].tolist()
bars2 = ax2.barh(docs, chunks, color=['#ff6b6b', '#4ecdc4'], height=0.4, edgecolor='#333')
for bar, val in zip(bars2, chunks):
    ax2.text(bar.get_width() + 0.1, bar.get_y() + bar.get_height()/2,
             f'{val} chunks', va='center', color='white', fontsize=9)
ax2.set_title('Estimated Vector Chunks per Document', color='white')
ax2.tick_params(colors='white')
ax2.set_yticklabels([f.replace('.md', '') for f in docs], color='white', fontsize=8)
for s in ax2.spines.values(): s.set_color('#333')

plt.tight_layout()
plt.show()

## 🔴 Upload Files → Create Vector Store → FileSearchTool

In `azure-ai-projects>=2.0.0`, vector stores are managed via **`openai_client.vector_stores`** (not `agents_client`):

```python
# New pattern
vector_store = openai_client.vector_stores.create(name='soc-kb')
openai_client.vector_stores.files.upload_and_poll(vector_store_id=vector_store.id, file=f)
tool = FileSearchTool(vector_store_ids=[vector_store.id])
```


In [ ]:
if not MOCK_MODE:
    from azure.ai.projects import AIProjectClient
    from azure.ai.projects.models import PromptAgentDefinition, FileSearchTool
    from azure.identity import DefaultAzureCredential

    credential = DefaultAzureCredential()
    project_client = AIProjectClient(endpoint=AZURE_AI_PROJECT_ENDPOINT, credential=credential)
    openai_client = project_client.get_openai_client()

    # Create vector store via openai_client (new SDK — no longer uses agents_client.vector_stores)
    vector_store = openai_client.vector_stores.create(name='soc-knowledge-base')
    print(f'✅ Vector store created: {vector_store.id}')

    # Upload both knowledge base files directly into the vector store
    for path in [playbook_path, threat_intel_path]:
        with open(path, 'rb') as f:
            vf = openai_client.vector_stores.files.upload_and_poll(
                vector_store_id=vector_store.id,
                file=f,
            )
        print(f'✅ Uploaded {path.name} → status: {vf.status}')

    print(f'\n✅ Vector store ready: {vector_store.id}')
else:
    print('⚠️  Skipping upload — set AZURE_AI_PROJECT_ENDPOINT\n')
    print('MOCK: Simulating vector store creation...')
    vector_store = type('vs', (), {'id': 'vs_mock_001'})()
    print(f'  vector_store.id = {vector_store.id}')


In [ ]:
if not MOCK_MODE:
    file_search = FileSearchTool(vector_store_ids=[vector_store.id])

    rag_agent = project_client.agents.create_version(
        agent_name='soc-knowledge-agent',
        definition=PromptAgentDefinition(
            model=MODEL_DEPLOYMENT_NAME,
            instructions=(
                'You are a SOC Knowledge Expert. You have access to a curated threat intelligence '
                'library and IR playbooks. When answering questions, always cite specific document '
                'sections. Use concrete details from the documents — never fabricate IOCs or TTPs. '
                'Format answers as structured JSON with keys: answer, source_sections, confidence.'
            ),
            tools=[file_search],
            tool_resources=file_search.resources,
        ),
    )
    print(f'✅ RAG Agent created: {rag_agent.name}')
else:
    print('⚠️  Skipping agent creation — set AZURE_AI_PROJECT_ENDPOINT')


## 🔴 Query the Knowledge Base

In [ ]:
soc_queries = [
    'What are the immediate containment steps for a Cobalt Strike beacon detection?',
    'Which MITRE ATT&CK techniques are associated with APT29?',
    'What KQL query should I use to hunt for credential dumping activity?',
]

if not MOCK_MODE:
    for query in soc_queries:
        response = openai_client.responses.create(
            input=query,
            extra_body={'agent_reference': {'name': rag_agent.name, 'type': 'agent_reference'}},
        )
        print(f'\n❓ Query: {query}')
        print('─' * 60)
        print(response.output_text[:600])

        # Extract file_citation annotations from the response output items
        for item in response.output:
            if hasattr(item, 'content'):
                for block in item.content:
                    if hasattr(block, 'annotations'):
                        for ann in block.annotations:
                            if ann.type == 'file_citation':
                                print(f'  📎 Citation: {getattr(ann, "filename", ann.file_citation.file_id)}')
else:
    print('⚠️  Skipping RAG query — set AZURE_AI_PROJECT_ENDPOINT\n')
    print('MOCK responses:')
    mock_answers = [
        {'q': soc_queries[0], 'a': '[From ir_playbook_credential_attack.md §Containment] 1. Isolate host. 2. Revoke service account tokens. 3. Memory dump before reboot. 4. Block C2 at firewall.', 'citation': 'ir_playbook_credential_attack.md'},
        {'q': soc_queries[1], 'a': '[From threat_intel_reference.md §APT Actors] APT29 uses: T1078 (Valid Accounts), T1555 (Credential Stores), T1071.001 (Web Protocols), T1105 (Ingress Tool Transfer).', 'citation': 'threat_intel_reference.md'},
        {'q': soc_queries[2], 'a': '[From ir_playbook_credential_attack.md §KQL Queries] SecurityEvent | where EventID == 4688 | where CommandLine has_any("mimikatz","sekurlsa","lsass") | project TimeGenerated, Computer, Account, CommandLine', 'citation': 'ir_playbook_credential_attack.md'},
    ]
    for item in mock_answers:
        print(f'\n❓ {item["q"]}')
        print(f'🤖 {item["a"]}')
        print(f'   📎 Citation: {item["citation"]}')


In [ ]:
# Simulate retrieval relevance scores for visualization (works in both modes)
retrieval_data = [
    {'Query': 'Cobalt Strike containment', 'Document': 'ir_playbook_credential_attack.md', 'Score': 0.91, 'Chunk': '§Immediate Containment Steps'},
    {'Query': 'Cobalt Strike containment', 'Document': 'threat_intel_reference.md',        'Score': 0.74, 'Chunk': '§Known Bad IPs'},
    {'Query': 'APT29 TTPs',               'Document': 'threat_intel_reference.md',        'Score': 0.95, 'Chunk': '§APT Actor Profiles'},
    {'Query': 'APT29 TTPs',               'Document': 'ir_playbook_credential_attack.md', 'Score': 0.62, 'Chunk': '§MITRE Mapping'},
    {'Query': 'KQL credential dump',      'Document': 'ir_playbook_credential_attack.md', 'Score': 0.88, 'Chunk': '§Detection KQL Queries'},
    {'Query': 'KQL credential dump',      'Document': 'threat_intel_reference.md',        'Score': 0.44, 'Chunk': '§IOC Patterns'},
]
df_retr = pd.DataFrame(retrieval_data)

fig, ax = plt.subplots(figsize=(12, 5))
fig.patch.set_facecolor('#0d1117')
ax.set_facecolor('#0d1117')

queries = df_retr['Query'].unique()
docs = df_retr['Document'].unique()
pivot = df_retr.pivot_table(index='Query', columns='Document', values='Score', fill_value=0)

sns.heatmap(pivot, ax=ax, cmap='YlOrRd', vmin=0, vmax=1, annot=True, fmt='.2f',
            linewidths=1, linecolor='#333',
            cbar_kws={'label': 'Relevance Score'},
            annot_kws={'size': 11, 'color': 'black'})

ax.set_title('RAG Retrieval Relevance Scores by Query × Document', color='white', pad=12)
ax.tick_params(colors='white')
ax.set_xlabel('Document', color='white')
ax.set_ylabel('Query', color='white')
ax.set_xticklabels([d.replace('.md', '') for d in pivot.columns], rotation=20, ha='right', color='white')
ax.set_yticklabels(pivot.index, rotation=0, color='white')
ax.axhline(y=len(queries), color='white', linewidth=0.5)

plt.tight_layout()
plt.show()

print('\n🎯 High relevance (>0.85): Used as grounding context')
print('⚠️  Medium relevance (0.65-0.85): Included with lower weight')
print('❌ Low relevance (<0.65): Filtered out to prevent hallucination')

## 🔴 Enterprise Scale: AzureAISearchTool

For production SOCs with Microsoft Sentinel, the `AzureAISearchTool` connects directly to your Azure AI Search index — which can contain millions of events, historical investigations, and threat reports.

In [ ]:
if not MOCK_MODE and SEARCH_ENABLED:
    from azure.ai.projects.models import (
        AzureAISearchTool, AzureAISearchToolResource,
        AISearchIndexResource, AzureAISearchQueryType,
    )

    ai_search_tool = AzureAISearchTool(
        azure_ai_search=AzureAISearchToolResource(
            indexes=[
                AISearchIndexResource(
                    project_connection_id=AZURE_AI_SEARCH_CONNECTION_NAME,
                    index_name=AZURE_AI_SEARCH_INDEX_NAME,
                    query_type=AzureAISearchQueryType.SIMPLE,
                )
            ]
        )
    )

    enterprise_agent = project_client.agents.create_version(
        agent_name='soc-enterprise-search',
        definition=PromptAgentDefinition(
            model=MODEL_DEPLOYMENT_NAME,
            instructions=(
                'You are a SOC Enterprise Analyst with access to the full SIEM history via Azure AI Search. '
                'Search for related historical incidents, IOC matches, and behavioral patterns. '
                'Always return structured JSON with: matching_cases, common_ttps, recommended_response.'
            ),
            tools=[ai_search_tool],
            tool_resources=ai_search_tool.resources,
        ),
    )

    response = openai_client.responses.create(
        input='Has IP 194.165.16.101 appeared in any historical incidents? What was the outcome?',
        extra_body={'agent_reference': {'name': enterprise_agent.name, 'type': 'agent_reference'}},
    )
    print('🏢 Enterprise Search Result:')
    print(response.output_text)
    project_client.agents.delete_version(agent_name=enterprise_agent.name, agent_version=enterprise_agent.version)
elif MOCK_MODE:
    print('⚠️  Skipping — set AZURE_AI_PROJECT_ENDPOINT + AZURE_AI_SEARCH_CONNECTION_NAME')
else:
    print('⚠️  Azure AI Search not configured — set AZURE_AI_SEARCH_CONNECTION_NAME')


In [ ]:
# Tool comparison: FileSearch vs AzureAISearch
comparison_data = [
    {'Dimension': 'Max Documents',    'FileSearchTool': '10,000 files',           'AzureAISearchTool': 'Millions (no limit)'},
    {'Dimension': 'Max File Size',    'FileSearchTool': '512 MB per file',        'AzureAISearchTool': 'Indexed (any)'},
    {'Dimension': 'Index Type',       'FileSearchTool': 'Foundry-managed vector', 'AzureAISearchTool': 'Your Azure AI Search'},
    {'Dimension': 'Setup Effort',     'FileSearchTool': 'Upload & auto-index',    'AzureAISearchTool': 'Pre-built index required'},
    {'Dimension': 'Data Source',      'FileSearchTool': 'Files (PDF, MD, DOCX...)', 'AzureAISearchTool': 'Any indexed data (SIEM, OneLake)'},
    {'Dimension': 'Best For',         'FileSearchTool': 'Playbooks, runbooks, docs', 'AzureAISearchTool': 'Historical SIEM data, investigation history'},
    {'Dimension': 'Cost Model',       'FileSearchTool': 'Per token (Foundry)',    'AzureAISearchTool': 'Azure AI Search pricing'},
    {'Dimension': 'Citation Support', 'FileSearchTool': 'file_citation_annotations', 'AzureAISearchTool': 'Source document reference'},
]

df_compare = pd.DataFrame(comparison_data)
display(df_compare.style
        .set_properties(**{'background-color': '#1e2127', 'color': '#e6e6e6', 'text-align': 'left'})
        .map(lambda v: 'color: #4ecdc4; font-weight: bold' if v and 'FileSearch' in str(v) else '', subset=['FileSearchTool'])
        .map(lambda v: 'color: #ff6b6b; font-weight: bold' if v and 'Azure' in str(v) else '', subset=['AzureAISearchTool'])
        .hide(axis='index'))

In [ ]:
# Cleanup
if not MOCK_MODE:
    project_client.agents.delete_version(agent_name=rag_agent.name, agent_version=rag_agent.version)
    openai_client.vector_stores.delete(vector_store.id)
    openai_client.close()
    project_client.close()
    print('✅ Cleanup complete: agent and vector store deleted')
else:
    print('⚠️  Nothing to clean up in mock mode')
